# Kelvin--Helmholtz passive-dye benchmark, FAST incompressible run

This clean notebook reuses the Kelvin--Helmholtz setup from `KelvinHelmholtz.ipynb` while defaulting to a local FAST run. It uses the reduced-MHD hydrodynamic limit (`psi=0`) and a passive dye following the smooth double-shear benchmark style used by Lecoanet et al.

The production-scale settings are shown below but are intentionally not the default.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from mhx.benchmarks.kelvin_helmholtz import (
    KelvinHelmholtzConfig,
    dye_entropy,
    kelvin_helmholtz_dye_rhs,
    kelvin_helmholtz_grid,
    kelvin_helmholtz_initial_state_from_config,
    run_kelvin_helmholtz_dye,
)
from mhx.runtime import configure_jax

configure_jax(enable_x64=True)

OUTPUT_ROOT = Path(os.environ.get("MHX_EXAMPLE_OUTDIR_ROOT", "outputs/examples"))
OUTDIR = OUTPUT_ROOT / "kelvin_helmholtz_incompressible"
OUTDIR.mkdir(parents=True, exist_ok=True)
print(f"Writing outputs to {OUTDIR.resolve()}")

Writing outputs to /mnt/c/Users/somar/Desktop/PhotosnVideos/cod/MHX/examples/outputs/examples/kelvin_helmholtz_incompressible


## Inputs

The FAST defaults are deliberately small enough to run locally. For a larger exploratory run, change `shape`, `t_end`, and `save_every`; for a production-quality hydrodynamic benchmark, use much higher resolution and document convergence separately.

In [6]:
FAST_CONFIG = KelvinHelmholtzConfig(
    shape=(32, 64),
    dt=2.0e-3,
    t_end=0.2,
    save_every=20,
    viscosity=1.0e-3,
    perturbation_amplitude=1.0e-2,
)

PRODUCTION_HINT = KelvinHelmholtzConfig(
    shape=(512, 1024),
    dt=1.0e-3,
    t_end=10.0,
    save_every=200,
    viscosity=1.0e-6,
    perturbation_amplitude=1.0e-2,
)

config = FAST_CONFIG
print(config)

KelvinHelmholtzConfig(shape=(512, 1024), lower=(0.0, 0.0), upper=(1.0, 2.0), viscosity=1e-06, resistivity=0.0, dt=0.001, t_end=10.0, save_every=200, shear_width=0.05, perturbation_width=0.2, perturbation_amplitude=0.01, flow_speed=1.0, y1=0.5, y2=1.5)


## Inspect the initial condition and RHS

The notebook intentionally exposes the main building blocks rather than hiding everything behind one driver call.

In [ ]:
grid = kelvin_helmholtz_grid(config)
state0 = kelvin_helmholtz_initial_state_from_config(grid, config)
rhs0 = kelvin_helmholtz_dye_rhs(
    state0,
    params=run_kelvin_helmholtz_dye(config).params,
    lengths=grid.lengths,
)

print("grid shape:", grid.shape)
print("initial dye entropy:", float(dye_entropy(state0.dye, grid)))
omega0 = np.asarray(state0.mhd.omega)
print("initial omega range:", float(np.min(omega0)), float(np.max(omega0)))
print("RHS dye max norm:", float(np.max(np.abs(np.asarray(rhs0.dye)))))

## Run the simulation

In [ ]:
result = run_kelvin_helmholtz_dye(config)
result.trajectory.times.block_until_ready()

print("saved times:", np.asarray(result.trajectory.times))
print("entropy history:", np.asarray(result.entropy))
final_dye_array = np.asarray(result.final_state.dye)
print("final dye range:", float(np.min(final_dye_array)), float(np.max(final_dye_array)))

## Plot outputs

In [5]:
extent = (config.lower[0], config.upper[0], config.lower[1], config.upper[1])
initial_dye = np.asarray(state0.dye)
final_dye = np.asarray(result.final_state.dye)
final_omega = np.asarray(result.final_state.mhd.omega)

fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
images = [
    axes[0].imshow(initial_dye.T, origin="lower", extent=extent, cmap="RdBu_r", vmin=0.0, vmax=1.0),
    axes[1].imshow(final_dye.T, origin="lower", extent=extent, cmap="RdBu_r", vmin=0.0, vmax=1.0),
]
axes[0].set_title(f"Initial Dye Concentration, t={float(result.trajectory.times[0]):.3f}")
axes[1].set_title(f"Final Dye Concentration, t={float(result.trajectory.times[-1]):.3f}")
for ax, image in zip(axes[:2], images, strict=True):
    ax.set_xlabel("x/Lx")
    ax.set_ylabel("y/Ly")
    fig.colorbar(image, ax=ax, shrink=0.8, label="Concentration (c)")
vort_im = axes[2].imshow(final_omega.T, origin="lower", extent=extent, cmap="magma")
axes[2].set_title(f"Final Vorticity, t={float(result.trajectory.times[-1]):.3f}")
axes[2].set_xlabel("x/Lx")
axes[2].set_ylabel("y/Ly")
fig.colorbar(vort_im, ax=axes[2], shrink=0.8, label="Vorticity")
fig.savefig(OUTDIR / "kh_incompressible_snapshots.png", dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(5, 3), constrained_layout=True)
ax.plot(np.asarray(result.trajectory.times), np.asarray(result.entropy), marker="o")
ax.set_xlabel("time")
ax.set_ylabel("dye entropy")
ax.set_title("Kelvin--Helmholtz dye entropy")
ax.grid(alpha=0.3)
fig.savefig(OUTDIR / "kh_incompressible_entropy.png", dpi=180)
plt.close(fig)

print("wrote", OUTDIR / "kh_incompressible_snapshots.png")
print("wrote", OUTDIR / "kh_incompressible_entropy.png")

wrote outputs/examples/kelvin_helmholtz_incompressible/kh_incompressible_snapshots.png
wrote outputs/examples/kelvin_helmholtz_incompressible/kh_incompressible_entropy.png
